# InstaNovo + InstaNovo+ de novo predictions — Ecoli_EV_2 with FINETUNED models

Mirrors `instanovo_colab_ecoli.ipynb` but uses the fine-tuned
checkpoints (`model_finetune/{instanovo,instanovoplus}/model_best.ckpt`,
output of the respective finetune notebooks) and runs only on
**Ecoli_EV_2** (the held-out test fraction; Ecoli_EV_1 was used for
fine-tuning).

Runs both stages of the InstaNovo pipeline:
1. InstaNovo (transformer) predicts directly on the spectra.
2. InstaNovo+ (diffusion) refines those predictions, using
   `refinement_path=` pointing at the InstaNovo CSVs from stage 1.

Outputs six CSVs total (3 per tool) for the Jetson-side FDR pipeline.
Dir naming matches `run_conversions_finetune.sh` and the Casanovo
finetune layout — `_finetune` comes BEFORE `_mgf` / `_mgf_decoy`:

InstaNovo:
- `result_finetune/instanovo/ecoli/Ecoli_EV_2.csv`             (mzML)
- `result_finetune_mgf/instanovo/ecoli/Ecoli_EV_2.csv`         (MGF)
- `result_finetune_mgf_decoy/instanovo/ecoli/Ecoli_EV_2.decoy.csv` (decoy MGF)

InstaNovo+:
- `result_finetune/instanovoplus/ecoli/Ecoli_EV_2.csv`             (mzML)
- `result_finetune_mgf/instanovoplus/ecoli/Ecoli_EV_2.csv`         (MGF)
- `result_finetune_mgf_decoy/instanovoplus/ecoli/Ecoli_EV_2.decoy.csv` (decoy MGF)


In [1]:
!nvidia-smi

Sat May  2 18:33:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Install dependencies

In [2]:
try:
  import instanovo
  !instanovo version
except ImportError:
  !pip install "instanovo[cu126]>=1.2.2" pyopenms-viz
  print('Installation complete. Restarting runtime to apply changes...')
  import os
  os.kill(os.getpid(), 9)

┏━━━━━━━━━━━━┳━━━━━━━━━┓
┃ Package    ┃ Version ┃
┡━━━━━━━━━━━━╇━━━━━━━━━┩
│ InstaNovo  │ 1.2.2   │
│ InstaNovo+ │ 1.2.2   │
│ NumPy      │ 2.2.6   │
│ PyTorch    │ 2.8.0   │
└────────────┴─────────┘


## Sync inputs from Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/data/ecoli
!mkdir -p /content/data_mgf/ecoli
!mkdir -p /content/data_mgf_decoy/ecoli
!mkdir -p /content/model_finetune/instanovo
!mkdir -p /content/model_finetune/instanovoplus
!mkdir -p /content/result_finetune/instanovo/ecoli
!mkdir -p /content/result_finetune_mgf/instanovo/ecoli
!mkdir -p /content/result_finetune_mgf_decoy/instanovo/ecoli
!mkdir -p /content/result_finetune/instanovoplus/ecoli
!mkdir -p /content/result_finetune_mgf/instanovoplus/ecoli
!mkdir -p /content/result_finetune_mgf_decoy/instanovoplus/ecoli

!mkdir -p /content/drive/MyDrive/DL-Project/result_finetune/instanovo/ecoli
!mkdir -p /content/drive/MyDrive/DL-Project/result_finetune_mgf/instanovo/ecoli
!mkdir -p /content/drive/MyDrive/DL-Project/result_finetune_mgf_decoy/instanovo/ecoli
!mkdir -p /content/drive/MyDrive/DL-Project/result_finetune/instanovoplus/ecoli
!mkdir -p /content/drive/MyDrive/DL-Project/result_finetune_mgf/instanovoplus/ecoli
!mkdir -p /content/drive/MyDrive/DL-Project/result_finetune_mgf_decoy/instanovoplus/ecoli

# Inputs — Ecoli_EV_2 only (held-out test)
!cp /content/drive/MyDrive/DL-Project/data/ecoli/Ecoli_EV_2.mzML            /content/data/ecoli/
!cp /content/drive/MyDrive/DL-Project/data_mgf/ecoli/Ecoli_EV_2.mgf         /content/data_mgf/ecoli/
!cp /content/drive/MyDrive/DL-Project/data_mgf_decoy/ecoli/Ecoli_EV_2.decoy.mgf /content/data_mgf_decoy/ecoli/

# Fine-tuned ckpts (outputs of the finetune notebooks)
!cp /content/drive/MyDrive/DL-Project/model_finetune/instanovo/model_best.ckpt    /content/model_finetune/instanovo/
!cp /content/drive/MyDrive/DL-Project/model_finetune/instanovoplus/model_best.ckpt /content/model_finetune/instanovoplus/

!ls -lh /content/data/ecoli/
!ls -lh /content/data_mgf/ecoli/
!ls -lh /content/data_mgf_decoy/ecoli/
!ls -lh /content/model_finetune/instanovo/
!ls -lh /content/model_finetune/instanovoplus/


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
total 985M
-rw------- 1 root root 985M May  2 18:34 Ecoli_EV_2.mzML
total 220M
-rw------- 1 root root 220M May  2 18:34 Ecoli_EV_2.mgf
total 232M
-rw------- 1 root root 232M May  2 18:34 Ecoli_EV_2.decoy.mgf
total 724M
drwxr-xr-x 4 root root 4.0K May  2 17:48 accelerator_state
-rw-r--r-- 1 root root 362M May  2 18:34 model_best.ckpt
-rw-r--r-- 1 root root 362M May  2 17:48 model_latest.ckpt
total 1.3G
-rw-r--r-- 1 root root 659M May  2 18:34 model_best.ckpt
-rw-r--r-- 1 root root 659M May  2 17:57 model_latest.ckpt


E Coli Instanovo

In [4]:
import os

input_path = "/content/data/ecoli"
output_path = "/content/result_finetune/instanovo/ecoli"
model_path = "/content/model_finetune/instanovo/model_best.ckpt"
samples = ["Ecoli_EV_2"]

for sample in samples:
    in_file = f"{input_path}/{sample}.mzML"
    out_file = f"{output_path}/{sample}.csv"
    if os.path.exists(out_file):
        print(f"Skipping {sample} (already exists)")
        continue
    !instanovo transformer predict --data-path {in_file} --output-path {out_file} --instanovo-model {model_path} num_workers=4 batch_size=512

!cp -r /content/result_finetune/instanovo/ecoli/. /content/drive/MyDrive/DL-Project/result_finetune/instanovo/ecoli

[05/02/26 18:34:22] INFO     Initializing InstaNovo inference.                                                                                                                 
[05/02/26 18:34:24] INFO     NumExpr defaulting to 12 threads.                                                                                                                 
2026-05-02 18:34:27.292266: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-02 18:34:27.363221: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
DEBUG:2026-05-02 18:34

E Coli MGF Instanovo

In [5]:
import os

input_path = "/content/data_mgf/ecoli"
output_path = "/content/result_finetune_mgf/instanovo/ecoli"
model_path = "/content/model_finetune/instanovo/model_best.ckpt"
samples = ["Ecoli_EV_2"]

for sample in samples:
    in_file = f"{input_path}/{sample}.mgf"
    out_file = f"{output_path}/{sample}.csv"
    if os.path.exists(out_file):
        print(f"Skipping {sample} (already exists)")
        continue
    !instanovo transformer predict --data-path {in_file} --output-path {out_file} --instanovo-model {model_path} num_workers=4 batch_size=512

!cp -r /content/result_finetune_mgf/instanovo/ecoli/. /content/drive/MyDrive/DL-Project/result_finetune_mgf/instanovo/ecoli


[05/02/26 18:43:46] INFO     Initializing InstaNovo inference.                                                                                                                 
[05/02/26 18:43:48] INFO     NumExpr defaulting to 12 threads.                                                                                                                 
2026-05-02 18:43:51.483202: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-02 18:43:51.553849: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
DEBUG:2026-05-02 18:43

E Coli MGF Decoy Instanovo

In [6]:
import os

input_path = "/content/data_mgf_decoy/ecoli"
output_path = "/content/result_finetune_mgf_decoy/instanovo/ecoli"
model_path = "/content/model_finetune/instanovo/model_best.ckpt"
samples = ["Ecoli_EV_2"]

for sample in samples:
    in_file = f"{input_path}/{sample}.decoy.mgf"
    out_file = f"{output_path}/{sample}.decoy.csv"
    if os.path.exists(out_file):
        print(f"Skipping {sample} (already exists)")
        continue
    !instanovo transformer predict --data-path {in_file} --output-path {out_file} --instanovo-model {model_path} num_workers=4 batch_size=512

!cp -r /content/result_finetune_mgf_decoy/instanovo/ecoli/. /content/drive/MyDrive/DL-Project/result_finetune_mgf_decoy/instanovo/ecoli


[05/02/26 18:52:18] INFO     Initializing InstaNovo inference.                                                                                                                 
[05/02/26 18:52:20] INFO     NumExpr defaulting to 12 threads.                                                                                                                 
2026-05-02 18:52:23.417848: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-02 18:52:23.487212: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
DEBUG:2026-05-02 18:52

E Coli Instanovo+ (mzML)

Refines the InstaNovo mzML predictions with the fine-tuned InstaNovo+ ckpt.
`refinement_path=` points at the InstaNovo CSV produced two cells above.

In [7]:
import os

input_path = "/content/data/ecoli"
output_path = "/content/result_finetune/instanovoplus/ecoli"
refinement_path = "/content/result_finetune/instanovo/ecoli"
model_path = "/content/model_finetune/instanovoplus/model_best.ckpt"
samples = ["Ecoli_EV_2"]

for sample in samples:
    in_file = f"{input_path}/{sample}.mzML"
    out_file = f"{output_path}/{sample}.csv"
    re_file = f"{refinement_path}/{sample}.csv"
    if os.path.exists(out_file):
        print(f"Skipping {sample} (already exists)")
        continue
    !instanovo diffusion predict --data-path {in_file} --output-path {out_file} --instanovo-plus-model {model_path} refinement_path={re_file} num_workers=4 batch_size=512

!cp -r /content/result_finetune/instanovoplus/ecoli/. /content/drive/MyDrive/DL-Project/result_finetune/instanovoplus/ecoli


[05/02/26 19:00:55] INFO     Initializing InstaNovo+ inference.                                                                                                                
[05/02/26 19:00:57] INFO     NumExpr defaulting to 12 threads.                                                                                                                 
2026-05-02 19:01:00.164740: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-02 19:01:00.234908: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
DEBUG:2026-05-02 19:01

E Coli MGF Instanovo+

Refines the InstaNovo MGF predictions with the fine-tuned InstaNovo+ ckpt.

In [8]:
import os

input_path = "/content/data_mgf/ecoli"
output_path = "/content/result_finetune_mgf/instanovoplus/ecoli"
refinement_path = "/content/result_finetune_mgf/instanovo/ecoli"
model_path = "/content/model_finetune/instanovoplus/model_best.ckpt"
samples = ["Ecoli_EV_2"]

for sample in samples:
    in_file = f"{input_path}/{sample}.mgf"
    out_file = f"{output_path}/{sample}.csv"
    re_file = f"{refinement_path}/{sample}.csv"
    if os.path.exists(out_file):
        print(f"Skipping {sample} (already exists)")
        continue
    !instanovo diffusion predict --data-path {in_file} --output-path {out_file} --instanovo-plus-model {model_path} refinement_path={re_file} num_workers=4 batch_size=512

!cp -r /content/result_finetune_mgf/instanovoplus/ecoli/. /content/drive/MyDrive/DL-Project/result_finetune_mgf/instanovoplus/ecoli


[05/02/26 19:06:36] INFO     Initializing InstaNovo+ inference.                                                                                                                
[05/02/26 19:06:38] INFO     NumExpr defaulting to 12 threads.                                                                                                                 
2026-05-02 19:06:41.085552: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-02 19:06:41.157840: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
DEBUG:2026-05-02 19:06

E Coli MGF Decoy Instanovo+

Refines the InstaNovo decoy-MGF predictions with the fine-tuned InstaNovo+ ckpt.

In [9]:
import os

input_path = "/content/data_mgf_decoy/ecoli"
output_path = "/content/result_finetune_mgf_decoy/instanovoplus/ecoli"
refinement_path = "/content/result_finetune_mgf_decoy/instanovo/ecoli"
model_path = "/content/model_finetune/instanovoplus/model_best.ckpt"
samples = ["Ecoli_EV_2"]

for sample in samples:
    in_file = f"{input_path}/{sample}.decoy.mgf"
    out_file = f"{output_path}/{sample}.decoy.csv"
    re_file = f"{refinement_path}/{sample}.decoy.csv"
    if os.path.exists(out_file):
        print(f"Skipping {sample} (already exists)")
        continue
    !instanovo diffusion predict --data-path {in_file} --output-path {out_file} --instanovo-plus-model {model_path} refinement_path={re_file} num_workers=4 batch_size=512

!cp -r /content/result_finetune_mgf_decoy/instanovoplus/ecoli/. /content/drive/MyDrive/DL-Project/result_finetune_mgf_decoy/instanovoplus/ecoli


[05/02/26 19:11:34] INFO     Initializing InstaNovo+ inference.                                                                                                                
[05/02/26 19:11:37] INFO     NumExpr defaulting to 12 threads.                                                                                                                 
2026-05-02 19:11:39.440883: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-02 19:11:39.512881: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
DEBUG:2026-05-02 19:11